# Proceso de automatizacion contable

In [1]:
# importar librerias a usar
import pandas as pd
from clase_reportes_new import ReportClassNew
import numpy as np
import tkinter as tk
from tkinter import filedialog
import json
import re
import os
import fitz  # PyMuPDF
from pathlib import Path
from collections import defaultdict
import holidays
rc = ReportClassNew()


In [2]:
# def informe_diario_mayoristas(
#             self,
#             ruta_carpeta: Optional[str] = None,
#             extension: str = 'csv',
#             producto_pen: Optional[List[str]] = None,
#             ruta_presupuesto: Optional[str] = None,
#             clientes: Optional[Dict[str, List[Dict[str, str]]]] = None,
#         ):
#         """
#         PENDIENTE       
#         """


In [3]:
ruta_carpeta = None
extension = 'csv'
producto_pen = None
ruta_presupuesto = None
clientes= None
    

In [4]:

# if ruta_carpeta:
#     ruta = Path(ruta_carpeta)
# else:
ruta = rc.validar_ruta() / 'CLEAN DATA'

df_ventas = rc.consolidar_carpeta(
    ruta_carpeta=ruta,
    extension=extension
)

Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA
  - Archivo 'Ventas_Junio_2024_Diciembre_2024.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2025_Diciembre_2025.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2026_Marzo_2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [5]:
#  def informe_cartera(self, categorias: list) -> pd.DataFrame:
#         """
#         """
#         # Base Cartera
        


In [6]:
categorias = [ 'MAYORISTA NV', 'FARMACIAS',
       'EXTERIOR', 'Surticosmeticos', 'COOPIDROGAS', 'Catalogo',
       'ESPECIALIZADAS', 'Distribuidor', 'KRIKA', 'HOLE COSMETICS',
      ]

In [7]:
# Archivo CSV de Google Sheets
url = "https://docs.google.com/spreadsheets/d/1uqGx-MkrUQR3znLq6HE2xiPts1RMv7P5GUQK4IJWdRQ/export?format=csv&gid=2087536586"

df = pd.read_csv(url)

ruta = rc.validar_ruta()
ruta_archivo = ruta / 'cartera' / 'Asiento contable (account.move).xlsx'
# Archivo Odoo
df_cartera = pd.read_excel(ruta_archivo)


# Archivo base cartera
df_cartera = df_cartera[df_cartera['Tipo de cliente'].isin(categorias)]

ruta_base = ruta / 'data' / 'base_cartera.xlsx'


In [8]:
df_cartera.columns

Index(['Tipo de Nota', 'Número', 'Nombre del contacto a mostrar en la factura',
       'Fecha de factura', 'Creado por', 'Fecha de vencimiento',
       'Tipo de cliente', 'Vendedor', 'Actividades', 'Factura de proveedor',
       'Importe sin impuestos firmado', 'Total firmado',
       'Total en divisa firmado', 'Medio de pago', 'Forma de Pago', 'TRM',
       'Importe pendiente firmado', 'Asesor Comercial', 'Payment Journals',
       'Estado de pago', 'Estado doc. DIAN', 'Estado'],
      dtype='object')

In [9]:

df_base_responsable = pd.read_excel(ruta_base, sheet_name='Responsables')
tipo_repetido = df_base_responsable[df_base_responsable['TIPO CLIENTE'].duplicated()]['TIPO CLIENTE'].unique()
tipo_repetido
responsables_tipo = df_base_responsable[~df_base_responsable['TIPO CLIENTE'].isin(tipo_repetido)]
responsables_clientes = df_base_responsable[df_base_responsable['TIPO CLIENTE'].isin(tipo_repetido)]
df_cartera = df_cartera.merge(responsables_tipo[['TIPO CLIENTE', 'RESPONSABLE', 'UBICACIÓN']], left_on='Tipo de cliente', right_on='TIPO CLIENTE', how='left' ).merge(responsables_clientes[[ 'CLIENTE', 'RESPONSABLE']], left_on='Nombre del contacto a mostrar en la factura', right_on='CLIENTE', how='left')
df_cartera['RESPONSABLE'] = df_cartera['RESPONSABLE_x'].fillna(df_cartera['RESPONSABLE_y'])

In [10]:
# df_cartera['TIPO CLIENTE'] = df_cartera['TIPO CLIENTE'].fillna(df_cartera['Tipo de cliente'])

In [11]:
responsable_default =df_base_responsable[df_base_responsable['TIPO CLIENTE'] == 'Default']['RESPONSABLE'].values[0]

df_cartera['RESPONSABLE'] = df_cartera['RESPONSABLE'].fillna(responsable_default)


df_cartera = df_cartera[
        (df_cartera['Fecha de factura'] >= '2025-01-01')&
        (df_cartera['Número'].str.startswith('F'))]



In [12]:
df_cartera.columns

Index(['Tipo de Nota', 'Número', 'Nombre del contacto a mostrar en la factura',
       'Fecha de factura', 'Creado por', 'Fecha de vencimiento',
       'Tipo de cliente', 'Vendedor', 'Actividades', 'Factura de proveedor',
       'Importe sin impuestos firmado', 'Total firmado',
       'Total en divisa firmado', 'Medio de pago', 'Forma de Pago', 'TRM',
       'Importe pendiente firmado', 'Asesor Comercial', 'Payment Journals',
       'Estado de pago', 'Estado doc. DIAN', 'Estado', 'TIPO CLIENTE',
       'RESPONSABLE_x', 'UBICACIÓN', 'CLIENTE', 'RESPONSABLE_y',
       'RESPONSABLE'],
      dtype='object')

In [13]:


responsables = df_base_responsable['RESPONSABLE'].unique().tolist()


df_cartera = df_cartera[['Número', 'Nombre del contacto a mostrar en la factura', 'Fecha de factura', 'Fecha de vencimiento', 'Importe pendiente firmado', 'RESPONSABLE', 'TIPO CLIENTE', 'Tipo de cliente']]
df_cartera['Dias de credito'] = (pd.to_datetime(df_cartera['Fecha de vencimiento']) - pd.to_datetime(df_cartera['Fecha de factura'])).dt.days    
# df_cartera = df_cartera[df_cartera['Dias de credito'] != 0]
df_cartera['Dias de atraso'] = (pd.to_datetime('now') - pd.to_datetime(df_cartera['Fecha de vencimiento'])  ).dt.days    
dias = df_cartera['Dias de atraso']

conditions = [
    dias < -7,                         # Más de 7 días antes de vencer
    dias.between(-7, 0, inclusive='both'),  # Próximo a vencer
    dias.between(1, 10, inclusive='both'),  # Hasta 10 días vencido
    dias.between(11, 30, inclusive='both'),
    dias.between(31, 60, inclusive='both'),
    dias.between(61, 90, inclusive='both'),
    dias > 90
]

choices = [
    'Corriente',
    'Proximo',
    'Corriente',
    '11_30',
    '31_60',
    '61_90',
    '90+'
]

df_cartera['Rango Mora'] = np.select(conditions, choices, default='Sin clasificar')
# Impervinculo a formulario de Google Forms para cada Factura
df_cartera['Link_forms'] = df_cartera.apply( 
    lambda row: (f'=HIPERVINCULO("https://docs.google.com/forms/d/e/1FAIpQLSfGupw7MUupqTAZr61Qgk6UcEPuJfZsKci9yBaahwOTVfGQ-Q/viewform?usp=pp_url&entry.1478832904={row['Nombre del contacto a mostrar en la factura']}&entry.1421149375={row['Número']}";"Link")'
    if row['Rango Mora'] not in ['Corriente', 'Proximo'] else ''), 
    axis=1
)

df_cartera.sort_values(by=['Nombre del contacto a mostrar en la factura', 'Fecha de vencimiento'], ascending=True, inplace=True)

df_cartera.rename(columns={'Nombre del contacto a mostrar en la factura': 'CLIENTE', 
                        'Fecha de factura': 'FECHA FACTURA', 'Fecha de vencimiento': 'FECHA VENCIMIENTO',
                            'Importe pendiente firmado': 'IMPORTE PENDIENTE', 'RESPONSABLE': 'RESPONSABLE', 
                            'Dias de credito': 'DIAS CREDITO', 'Dias de atraso': 'DIAS ATRASO', 
                            'Rango Mora': 'RANGO MORA', 'Link_forms': 'LINK FORMS'}, inplace=True)



In [16]:
df_cartera[df_cartera['CLIENTE']=='COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS']

,Número,CLIENTE,FECHA FACTURA,FECHA VENCIMIENTO,IMPORTE PENDIENTE,RESPONSABLE,TIPO CLIENTE,Tipo de cliente,DIAS CREDITO,DIAS ATRASO,RANGO MORA,LINK FORMS
44,FEVY78973,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-10-29,2025-12-13,3.429283e+05,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,96,90+,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
45,FEVY78969,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-10-29,2025-12-13,8.328441e+05,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,96,90+,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
43,FEVY81950,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-11-13,2025-12-28,4.371534e+05,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,81,61_90,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
31,FEVY101535,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,2.343300e+04,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
32,FEVY101532,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,2.919885e+05,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
33,FEVY101527,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,6.556500e+04,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
34,FEVY101526,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,2.882416e+04,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
35,FEVY101517,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,5.226765e+06,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
36,FEVY101510,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,1.837806e+06,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."
37,FEVY101507,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,2025-12-24,2026-02-07,1.866872e+05,DIANA RIOS,COOPIDROGAS,COOPIDROGAS,45,40,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d..."


In [14]:

# Genera los archivos CSV para cada responsable
for i in responsables:
    if df_cartera['RESPONSABLE'].shape[0] > 1:
        df_cartera[df_cartera['RESPONSABLE'] == i][['Número', 'CLIENTE', 'FECHA FACTURA', 'FECHA VENCIMIENTO',
       'IMPORTE PENDIENTE', 'RESPONSABLE', 'Tipo de cliente',
       'DIAS CREDITO', 'DIAS ATRASO', 'RANGO MORA', 'LINK FORMS']].to_csv(ruta / 'cartera' / f'CARTERA_{i}.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')


In [31]:
df_cartera['id_tipo'] = df_cartera['RESPONSABLE'] + df_cartera['TIPO CLIENTE']

,Número,CLIENTE,FECHA FACTURA,FECHA VENCIMIENTO,IMPORTE PENDIENTE,RESPONSABLE,TIPO CLIENTE,DIAS CREDITO,DIAS ATRASO,RANGO MORA,LINK FORMS,id_tipo
63,FE9007,AMAYA AMAYA DIANA MARCELA,2026-03-16,2026-03-16,5848248,MIRIAM BURGOS,MAYORISTA NV,0,3,Corriente,,MIRIAM BURGOSMAYORISTA NV
115,FE4638,AMINTA INES ROMERO SOTELO,2026-02-26,2026-03-28,9680745,MIRIAM BURGOS,Distribuidor,30,-9,Corriente,,MIRIAM BURGOSDistribuidor
126,FE3663,BELLPROF GROUP SAS,2026-02-20,2026-03-22,1769399,MIRIAM BURGOS,Distribuidor,30,-3,Proximo,,MIRIAM BURGOSDistribuidor
99,FE6415,BELLPROF GROUP SAS,2026-03-05,2026-04-04,41159704,MIRIAM BURGOS,Distribuidor,30,-16,Corriente,,MIRIAM BURGOSDistribuidor
155,FE2,BRECCIA SALUD S.AS.,2026-01-28,2026-02-27,32715421,DIANA RIOS,NaN,30,20,11_30,"=HIPERVINCULO(""https://docs.google.com/forms/d...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
39,FYEX27,ZAR IMPORT ZARIMPORT S.A.,2025-12-15,2025-12-30,45462360,SHELLSY VELASCO,EXTERIOR,15,79,61_90,"=HIPERVINCULO(""https://docs.google.com/forms/d...",SHELLSY VELASCOEXTERIOR
30,FYEX33,ZAR IMPORT ZARIMPORT S.A.,2025-12-30,2026-01-14,278471960,SHELLSY VELASCO,EXTERIOR,15,64,61_90,"=HIPERVINCULO(""https://docs.google.com/forms/d...",SHELLSY VELASCOEXTERIOR
14,FYEX34,ZAR IMPORT ZARIMPORT S.A.,2026-01-28,2026-02-12,186022470,SHELLSY VELASCO,EXTERIOR,15,35,31_60,"=HIPERVINCULO(""https://docs.google.com/forms/d...",SHELLSY VELASCOEXTERIOR
4,FYEX36,ZAR IMPORT ZARIMPORT S.A.,2026-02-13,2026-02-28,15869776,SHELLSY VELASCO,EXTERIOR,15,19,11_30,"=HIPERVINCULO(""https://docs.google.com/forms/d...",SHELLSY VELASCOEXTERIOR


In [29]:

df_cartera['IMPORTE PENDIENTE'] = df_cartera['IMPORTE PENDIENTE'].fillna(0).astype(int)
df_cartera_pivot = df_cartera.pivot_table(index=['id_tipo','TIPO CLIENTE','CLIENTE','RESPONSABLE'], columns='RANGO MORA', values='IMPORTE PENDIENTE', aggfunc='sum', fill_value=0).reset_index()


In [ ]:
df_cartera_pivot = df_cartera_pivot[['id_tipo','TIPO CLIENTE','RESPONSABLE','CLIENTE', 'Corriente', 'Proximo', '11_30', '31_60', '61_90', '90+' ]]
df_cartera_pivot = df_cartera_pivot.copy()

In [30]:
df_cartera_pivot

RANGO MORA,id_tipo,TIPO CLIENTE,CLIENTE,RESPONSABLE,11_30,31_60,61_90,90+,Corriente,Proximo
0,ANDRES VASQUEZHOLE COSMETICS,HOLE COSMETICS,HOLE COSMETICS SAS,ANDRES VASQUEZ,0,0,0,0,169626506,0
1,DIANA RIOSCOOPIDROGAS,COOPIDROGAS,COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS,DIANA RIOS,6857135,45937634,437153,1175772,592997743,0
2,DIANA RIOSCatalogo,Catalogo,NOVAVENTA S.A.S,DIANA RIOS,0,0,0,0,2761628508,230129741
3,DIANA RIOSESPECIALIZADAS,ESPECIALIZADAS,CMX SAS,DIANA RIOS,0,0,0,0,27337817,6798117
4,DIANA RIOSESPECIALIZADAS,ESPECIALIZADAS,LASKIN S.A,DIANA RIOS,11023639,5724659,0,0,8414588,0
5,MIRIAM BURGOSDistribuidor,Distribuidor,AMINTA INES ROMERO SOTELO,MIRIAM BURGOS,0,0,0,0,9680745,0
6,MIRIAM BURGOSDistribuidor,Distribuidor,BELLPROF GROUP SAS,MIRIAM BURGOS,0,0,0,0,41159704,1769399
7,MIRIAM BURGOSDistribuidor,Distribuidor,CATALINA VIVAS SAS,MIRIAM BURGOS,0,0,0,0,38192149,0
8,MIRIAM BURGOSDistribuidor,Distribuidor,MARIA FERNANDA DUARTE GARCIA,MIRIAM BURGOS,0,0,0,0,29999940,0
9,MIRIAM BURGOSDistribuidor,Distribuidor,MVH INVERSIONES SAS,MIRIAM BURGOS,26007831,0,0,0,77792070,29945248


In [21]:
df_cartera['RESPONSABLE'].value_counts()

RESPONSABLE
DIANA RIOS         63
MIRIAM BURGOS      47
SHELLSY VELASCO    36
MARIA PAULA        10
ANDRES VASQUEZ      1
Name: count, dtype: int64

In [16]:
df_cartera_pivot['TOTAL'] = df_cartera_pivot[
    ['Corriente', 'Proximo', '11_30', '31_60', '61_90', '90+']
].sum(axis=1)



In [18]:
df_cartera_pivot['RESPONSABLE'].value_counts()

RESPONSABLE
MIRIAM BURGOS      28
SHELLSY VELASCO     6
DIANA RIOS          4
ANDRES VASQUEZ      1
Name: count, dtype: int64

In [ ]:

df_cartera_pivot.columns = df_cartera_pivot.columns.str.upper()
df_cartera_pivot_grouped = df_cartera_pivot.groupby(['TIPO CLIENTE','RESPONSABLE']).agg({
    'CLIENTE':'count',
    'CORRIENTE': 'sum',
    'PROXIMO': 'sum',
    '11_30': 'sum',
    '31_60': 'sum',
    '61_90': 'sum',
    '90+': 'sum',
    'TOTAL': 'sum'
}).reset_index()
df_cartera_pivot_grouped.sort_values(by='TOTAL', ascending=False, inplace=True)

from IPython.display import display, HTML
from datetime import datetime
import locale

# Configurar locale en español
locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')

fecha = datetime.today().strftime('%d de %B de %Y')


# ── HELPERS ───────────────────────────────────────────────────────────────────
def get_status_class(row):
    critico = row['90+'] + row['61_90']
    medio   = row['31_60'] + row['11_30']
    if critico > 0:
        return "background:#fff0f0;color:#c0392b;"
    elif medio > 0:
        return "background:#fffbeb;color:#b45309;"
    else:
        return "background:#f0faf4;color:#15803d;"

def fmt(v):
    return f"${v:,.0f}"


# ── INFORME POR RESPONSABLE ───────────────────────────────────────────────────
def build_email_html(responsable, df):
    t = df[['CORRIENTE','PROXIMO','11_30','31_60','61_90','90+','TOTAL']].sum()

    filas = ""
    for _, row in df.iterrows():
        total_style = get_status_class(row)
        filas += f"""
        <tr>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;font-weight:600;color:#1a1f2e;white-space:nowrap;">{row['TIPO CLIENTE'].upper()}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#2e7d32;white-space:nowrap;">{fmt(row['CORRIENTE'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#0077b6;white-space:nowrap;">{fmt(row['PROXIMO'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#e65100;white-space:nowrap;">{fmt(row['11_30'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#c62828;white-space:nowrap;">{fmt(row['31_60'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#b71c1c;white-space:nowrap;">{fmt(row['61_90'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;font-weight:700;color:#7b1fa2;white-space:nowrap;">{fmt(row['90+'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;font-weight:700;border-radius:4px;white-space:nowrap;{total_style}">{fmt(row['TOTAL'])}</td>
        </tr>"""

    iniciales = "".join([p[0] for p in responsable.split()[:2]]).upper()
    n_clientes = df['CLIENTE'].sum() if 'CLIENTE' in df.columns else "—"

    if t['TOTAL'] > 0:
        icv_30 = (t['31_60'] + t['61_90'] + t['90+']) / t['TOTAL'] * 100
        icv_90 = t['90+'] / t['TOTAL'] * 100
    else:
        icv_30 = icv_90 = 0

    html = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif;max-width:1100px;margin:0 auto;
                background:#ffffff;border-radius:12px;overflow:hidden;
                box-shadow:0 4px 24px rgba(0,0,0,0.08);">

    <!-- HEADER -->
    <div style="background:linear-gradient(135deg,#0f2044 0%,#1a3a6e 60%,#1e4d8c 100%);padding:28px 36px;">
        <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
            <td>
            <div style="color:#ffffff;font-size:20px;font-weight:700;">📋 Informe de Cartera</div>
            <div style="color:#93afd4;font-size:13px;margin-top:4px;">Estado de cuentas y vencimientos</div>
            </td>
            <td align="right">
            <span style="background:rgba(255,255,255,0.12);border:1px solid rgba(255,255,255,0.2);
                        color:#cddff5;padding:6px 14px;border-radius:20px;font-size:12px;">
                📅 {fecha}
            </span>
            </td>
        </tr>
        </table>
        <div style="margin-top:18px;display:flex;gap:20px;flex-wrap:wrap;">
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#d5f5e3;border:1px solid #a9dfbf;margin-right:5px;"></span>Cartera sana
        </span>
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#fffbeb;border:1px solid #f9e79f;margin-right:5px;"></span>Mora media (11–60 días)
        </span>
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#fff0f0;border:1px solid #f5b7b1;margin-right:5px;"></span>Mora crítica (61+ días)
        </span>
        </div>
    </div>

    <!-- RESPONSABLE -->
    <div style="padding:24px 36px 8px;">
        <table cellpadding="0" cellspacing="0">
        <tr>
            <td>
            <div style="width:38px;height:38px;border-radius:50%;
                        background:linear-gradient(135deg,#1a3a6e,#1e4d8c);
                        color:white;font-size:14px;font-weight:700;
                        text-align:center;line-height:38px;">{iniciales}</div>
            </td>
            <td style="padding-left:12px;">
            <div style="font-size:15px;font-weight:700;color:#0f2044;">{responsable}</div>
            <div style="font-size:12px;color:#8492a6;">{n_clientes} clientes</div>
            </td>
            <td style="padding-left:24px;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;">ICV 30</div>
            <div style="font-size:18px;font-weight:700;color:#c62828;">{icv_30:.2f}%</div>
            </td>
            <td style="padding-left:24px;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;">ICV 90+</div>
            <div style="font-size:18px;font-weight:700;color:#7b1fa2;">{icv_90:.2f}%</div>
            </td>
        </tr>
        </table>
    </div>

    <!-- TABLA -->
    <div style="padding:12px 36px 28px;overflow-x:auto;">
        <table width="100%" cellpadding="0" cellspacing="0"
            style="border-collapse:collapse;font-size:11.5px;
                    border:1px solid #e8ecf2;border-radius:8px;overflow:hidden;">
        <thead>
            <tr style="background:#f7f9fc;">
            <th style="padding:11px 10px;text-align:left;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;">Canal</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">Corriente</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">Próximo</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">11–30 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">31–60 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">61–90 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">90+ d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;">Total</th>
            </tr>
        </thead>
        <tbody>
            {filas}
            <tr style="background:#0f2044;">
            <td style="padding:12px 10px;color:#e8f0fc;font-weight:700;font-size:13px;">SUBTOTAL</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['CORRIENTE'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['PROXIMO'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['11_30'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['31_60'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['61_90'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['90+'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#7dd3fc;font-weight:700;font-size:14px;white-space:nowrap;">{fmt(t['TOTAL'])}</td>
            </tr>
        </tbody>
        </table>
    </div>

    <!-- FOOTER -->
    <div style="background:#f7f9fc;padding:16px 36px;border-top:1px solid #e0e5ee;
                display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
        <span style="font-size:11.5px;color:#8492a6;">Generado automáticamente · Análisis de Datos </span>
        <span style="font-size:11.5px;color:#8492a6;">Valores en COP · Incluye todos los rangos de mora</span>
    </div>
    </div>
    """
    return html


# ── INFORME CONSOLIDADO ───────────────────────────────────────────────────────
def build_email_html_consolidado(df):
    t = df[['CORRIENTE','PROXIMO','11_30','31_60','61_90','90+','TOTAL']].sum()

    filas = ""
    for _, row in df.iterrows():
        total_style = get_status_class(row)
        filas += f"""
        <tr>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;font-weight:600;color:#1a1f2e;white-space:nowrap;">{row['TIPO CLIENTE'].upper()}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#2e7d32;white-space:nowrap;">{fmt(row['CORRIENTE'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#0077b6;white-space:nowrap;">{fmt(row['PROXIMO'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#e65100;white-space:nowrap;">{fmt(row['11_30'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#c62828;white-space:nowrap;">{fmt(row['31_60'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;color:#b71c1c;white-space:nowrap;">{fmt(row['61_90'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;font-weight:700;color:#7b1fa2;white-space:nowrap;">{fmt(row['90+'])}</td>
        <td style="padding:8px 10px;border-bottom:1px solid #eef0f5;text-align:right;font-weight:700;border-radius:4px;white-space:nowrap;{total_style}">{fmt(row['TOTAL'])}</td>
        </tr>"""

    n_responsables = df['RESPONSABLE'].nunique() if 'RESPONSABLE' in df.columns else "—"
    n_clientes     = df['CLIENTE'].sum() if 'CLIENTE' in df.columns else "—"

    if t['TOTAL'] > 0:
        icv_30 = (t['31_60'] + t['61_90'] + t['90+']) / t['TOTAL'] * 100
        icv_90 = t['90+'] / t['TOTAL'] * 100
    else:
        icv_30 = icv_90 = 0

    html = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif;max-width:1100px;margin:0 auto;
                background:#ffffff;border-radius:12px;overflow:hidden;
                box-shadow:0 4px 24px rgba(0,0,0,0.08);">

    <!-- HEADER -->
    <div style="background:linear-gradient(135deg,#0f2044 0%,#1a3a6e 60%,#1e4d8c 100%);padding:28px 36px;">
        <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
            <td>
            <div style="color:#ffffff;font-size:20px;font-weight:700;">📋 Informe de Cartera — Consolidado</div>
            <div style="color:#93afd4;font-size:13px;margin-top:4px;">Vista global · Todos los responsables</div>
            </td>
            <td align="right">
            <span style="background:rgba(255,255,255,0.12);border:1px solid rgba(255,255,255,0.2);
                        color:#cddff5;padding:6px 14px;border-radius:20px;font-size:12px;">
                📅 {fecha}
            </span>
            </td>
        </tr>
        </table>
        <div style="margin-top:18px;display:flex;gap:20px;flex-wrap:wrap;">
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#d5f5e3;border:1px solid #a9dfbf;margin-right:5px;"></span>Cartera sana
        </span>
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#fffbeb;border:1px solid #f9e79f;margin-right:5px;"></span>Mora media (11–60 días)
        </span>
        <span style="color:#93afd4;font-size:12px;">
            <span style="display:inline-block;width:10px;height:10px;border-radius:3px;background:#fff0f0;border:1px solid #f5b7b1;margin-right:5px;"></span>Mora crítica (61+ días)
        </span>
        </div>
    </div>

    <!-- MÉTRICAS RESUMEN -->
    <div style="padding:20px 36px 8px;">
    <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
        <td style="text-align:center;padding:12px 8px;background:#f7f9fc;border-radius:8px;border:1px solid #e0e5ee;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">Responsables</div>
            <div style="font-size:22px;font-weight:700;color:#0f2044;">{n_responsables}</div>
        </td>
        <td style="width:12px;"></td>
        <td style="text-align:center;padding:12px 8px;background:#f7f9fc;border-radius:8px;border:1px solid #e0e5ee;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">Clientes</div>
            <div style="font-size:22px;font-weight:700;color:#0f2044;">{n_clientes}</div>
        </td>
        <td style="width:12px;"></td>
        <td style="text-align:center;padding:12px 8px;background:#f7f9fc;border-radius:8px;border:1px solid #e0e5ee;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">Cartera Total</div>
            <div style="font-size:22px;font-weight:700;color:#0f2044;">{fmt(t['TOTAL'])}</div>
        </td>
        <td style="width:12px;"></td>
        <td style="text-align:center;padding:12px 8px;background:#fff0f0;border-radius:8px;border:1px solid #f5b7b1;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">ICV 30</div>
            <div style="font-size:22px;font-weight:700;color:#c62828;">{icv_30:.2f}%</div>
        </td>
        <td style="width:12px;"></td>
        <td style="text-align:center;padding:12px 8px;background:#fdf4ff;border-radius:8px;border:1px solid #e9d5ff;">
            <div style="font-size:11px;color:#8492a6;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">ICV 90+</div>
            <div style="font-size:22px;font-weight:700;color:#7b1fa2;">{icv_90:.2f}%</div>
        </td>
        </tr>
    </table>
    </div>
    <!-- TABLA -->
    <div style="padding:12px 36px 28px;overflow-x:auto;">
        <table width="100%" cellpadding="0" cellspacing="0"
            style="border-collapse:collapse;font-size:11.5px;
                    border:1px solid #e8ecf2;border-radius:8px;overflow:hidden;">
        <thead>
            <tr style="background:#f7f9fc;">
            <th style="padding:11px 10px;text-align:left;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;">Canal</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">Corriente</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">Próximo</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">11–30 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">31–60 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">61–90 d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;white-space:nowrap;">90+ d</th>
            <th style="padding:11px 10px;text-align:right;font-weight:600;color:#5a6478;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:2px solid #e0e5ee;">Total</th>
            </tr>
        </thead>
        <tbody>
            {filas}
            <tr style="background:#0f2044;">
            <td style="padding:12px 10px;color:#e8f0fc;font-weight:700;font-size:13px;">TOTAL GENERAL</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['CORRIENTE'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['PROXIMO'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['11_30'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['31_60'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['61_90'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#e8f0fc;font-weight:700;white-space:nowrap;">{fmt(t['90+'])}</td>
            <td style="padding:12px 10px;text-align:right;color:#7dd3fc;font-weight:700;font-size:14px;white-space:nowrap;">{fmt(t['TOTAL'])}</td>
            </tr>
        </tbody>
        </table>
    </div>

    <!-- FOOTER -->
    <div style="background:#f7f9fc;padding:16px 36px;border-top:1px solid #e0e5ee;
                display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
        <span style="font-size:11.5px;color:#8492a6;">Generado automáticamente · Análisis de Datos </span>
        <span style="font-size:11.5px;color:#8492a6;">Valores en COP · Incluye todos los rangos de mora</span>
    </div>
    </div>
    """
    return html


# ── EJECUCIÓN ─────────────────────────────────────────────────────────────────
informes_por_responsable = {}



html_consolidado = build_email_html_consolidado(df_cartera_pivot_grouped)
informes_por_responsable['__CONSOLIDADO__'] = html_consolidado


# POR RESPONSABLE
for responsable in df_cartera_pivot_grouped['RESPONSABLE'].unique():
    df = df_cartera_pivot_grouped[df_cartera_pivot_grouped['RESPONSABLE'] == responsable].copy()
    html = build_email_html(responsable, df)
    informes_por_responsable[responsable] = html

# return informes_por_responsable, df_cartera






In [18]:
df_cartera_pivot_grouped['RESPONSABLE'].unique()

array(['DIANA RIOS', 'SHELLSY VELASCO', 'MIRIAM BURGOS', 'ANDRES VASQUEZ'],
      dtype=object)

In [19]:
df_cartera_pivot_grouped[df_cartera_pivot_grouped['RESPONSABLE'] == 'MIRIAM BURGOS']

RANGO MORA,TIPO CLIENTE,RESPONSABLE,CLIENTE,CORRIENTE,PROXIMO,11_30,31_60,61_90,90+,TOTAL
2,Distribuidor,MIRIAM BURGOS,13,275320877,78893115,35718328,2345627,0,0,392277947
7,MAYORISTA NV,MIRIAM BURGOS,15,96218018,0,6869048,17592577,0,0,120679643
